In [ ]:
import os
import time
import numpy as np
import pandas as pd
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification

SEED = 42
EPOCHS, BATCH, MAXLEN, LR = 3, 16, 256, 2e-5
MODEL_NAME = "distilbert-base-uncased"

torch.manual_seed(SEED)
device = torch.device("mps")

In [ ]:
df = pd.read_csv("../data/processed/dataset_v2.csv", dtype=str, keep_default_na=False)
df = df[df["split"].isin(["train", "val"])].reset_index(drop=True)
assert set(df["split"]) == {"train", "val"}, "unexpected split values"
df["y"] = (df["final_label"] == "attack").astype(int)
train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df = df[df["split"] == "val"].reset_index(drop=True)
print("train rows:", len(train_df), "| val rows:", len(val_df))

tok = AutoTokenizer.from_pretrained(MODEL_NAME)

class PromptDataset(Dataset):
    def __init__(self, frame, tokenizer, max_len):
        texts = [t[:4000] for t in frame["text"]]
        encoded = tokenizer(texts, truncation=True, max_length=max_len)
        self.input_ids = encoded["input_ids"]
        self.labels = frame["y"].tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return {"input_ids": self.input_ids[i], "label": self.labels[i]}

In [ ]:

def collate(items):
    seqs = [torch.tensor(x["input_ids"]) for x in items]
    lengths = torch.tensor([len(s) for s in seqs])
    input_ids = pad_sequence(seqs, batch_first=True, padding_value=tok.pad_token_id)
    attention_mask = (torch.arange(input_ids.shape[1])[None, :] < lengths[:, None]).long()
    labels = torch.tensor([x["label"] for x in items])
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}


train_ds = PromptDataset(train_df, tok, MAXLEN)
val_ds = PromptDataset(val_df, tok, MAXLEN)

shuffle_gen = torch.Generator()
shuffle_gen.manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, collate_fn=collate, generator=shuffle_gen)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, collate_fn=collate)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

In [ ]:
def to_device(batch):
    return {k: v.to(device) for k, v in batch.items()}


def predict_probs(loader):
    model.eval()
    probs = []
    with torch.no_grad():
        for batch in loader:
            batch = to_device(batch)
            logits = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]).logits
            probs.extend(torch.softmax(logits, dim=-1)[:, 1].cpu().tolist())
    return np.array(probs)


best_f1, best_probs = -1.0, None
for epoch in range(1, EPOCHS + 1):
    model.train()
    t0 = time.time()
    running = 0.0
    for step, batch in enumerate(train_loader, start=1):
        batch = to_device(batch)
        out = model(**batch)
        out.loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        running += out.loss.item()
        if step % 100 == 0:
            print(f"epoch {epoch} step {step}: avg loss {running / 100:.3f} | {time.time() - t0:.0f}s")
            running = 0.0

    probs = predict_probs(val_loader)
    pred = (probs >= 0.5).astype(int)
    p = precision_score(val_df["y"], pred, zero_division=0)
    r = recall_score(val_df["y"], pred, zero_division=0)
    f = f1_score(val_df["y"], pred, zero_division=0)
    print(f"EPOCH {epoch} validation | attack precision {p:.3f} recall {r:.3f} F1 {f:.3f}")
    if f > best_f1:
        best_f1, best_probs = f, probs
        os.makedirs("../models/distilbert_v2", exist_ok=True)
        model.save_pretrained("../models/distilbert_v2")
        tok.save_pretrained("../models/distilbert_v2")
        print("  saved as the best epoch so far")

os.makedirs("../results/tables", exist_ok=True)
val_df["prob_attack"] = best_probs
val_df["pred"] = (best_probs >= 0.5).astype(int)
val_df[["id", "source_dataset", "y", "prob_attack"]].to_csv("../results/tables/distilbert_v2_val_probs.csv", index=False)

rows = []
for src, grp in val_df.groupby("source_dataset"):
    atk = grp[grp["y"] == 1]
    saf = grp[grp["y"] == 0]
    rows.append({
        "source_dataset": src,
        "attack_rows": len(atk),
        "attack_caught": float((atk["pred"] == 1).mean()) if len(atk) else float("nan"),
        "safe_rows": len(saf),
        "safe_wrongly_flagged": float((saf["pred"] == 1).mean()) if len(saf) else float("nan"),
    })
by_source = pd.DataFrame(rows).round(3)
by_source.to_csv("../results/tables/distilbert_v2_val_by_source.csv", index=False)

print()
print("BEST EPOCH, validation set")
print(classification_report(val_df["y"], val_df["pred"], target_names=["safe", "attack"], digits=3))
print(confusion_matrix(val_df["y"], val_df["pred"]))
print()
print(by_source.to_string(index=False))

In [ ]:
import numpy as np
import pandas as pd

v = pd.read_csv("../results/tables/distilbert_v2_val_probs.csv")
n_attack = int((v["y"] == 1).sum())
n_safe = int((v["y"] == 0).sum())

rows = []
for t in [0.3, 0.5, 0.7, 0.8, 0.9, 0.95, 0.98]:
    flagged = v["prob_attack"] >= t
    tp = int(((v["y"] == 1) & flagged).sum())
    fp = int(((v["y"] == 0) & flagged).sum())
    rows.append({
        "threshold": t,
        "attack_recall": round(tp / n_attack, 3),
        "attack_precision": round(tp / max(tp + fp, 1), 3),
        "safe_flagged": fp,
        "safe_flagged_pct": round(100 * fp / n_safe, 2),
    })
print(pd.DataFrame(rows).to_string(index=False))

cut = np.quantile(v.loc[v["y"] == 0, "prob_attack"], 0.99)
recall_at_1pct = (v.loc[v["y"] == 1, "prob_attack"] > cut).mean()
print()
print("threshold that flags about 1% of safe rows:", round(float(cut), 3))
print("attack recall at that threshold:", round(float(recall_at_1pct), 3))

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

df = pd.read_csv("../data/processed/dataset_v2.csv", dtype=str, keep_default_na=False)
df = df[df["split"].isin(["train", "val"])].reset_index(drop=True)
df["y"] = (df["final_label"] == "attack").astype(int)
df["short_text"] = df["text"].str[:3000]
train = df[df["split"] == "train"]
val = df[df["split"] == "val"].reset_index(drop=True)

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, max_features=200000)
X_train = vec.fit_transform(train["short_text"])
X_val = vec.transform(val["short_text"])
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, train["y"])

base = val[["id", "source_dataset", "y"]].copy()
base["prob_attack"] = lr.predict_proba(X_val)[:, 1]
base.to_csv("../results/tables/tfidf_lr_v2_val_probs.csv", index=False)

dis = pd.read_csv("../results/tables/distilbert_v2_val_probs.csv")
assert set(base["id"]) == set(dis["id"]), "the two models were scored on different rows"


def recall_at_fp_rate(frame, rate):
    cut = np.quantile(frame.loc[frame["y"] == 0, "prob_attack"], 1 - rate)
    return round(float((frame.loc[frame["y"] == 1, "prob_attack"] > cut).mean()), 3)


for name, frame in [("TF-IDF + logistic regression", base), ("DistilBERT (epoch 2)", dis)]:
    print(name)
    print("   average precision:", round(average_precision_score(frame["y"], frame["prob_attack"]), 3))
    for rate in [0.01, 0.005, 0.002]:
        print(f"   attack recall when {rate * 100:g}% of safe rows are flagged:", recall_at_fp_rate(frame, rate))